# Equiflow: Diagrama de Flujo de Cohorte

Este notebook genera un diagrama de flujo que muestra cómo evoluciona la composición del cohorte a través de los pasos de selección y preprocesamiento del pipeline, usando la librería [Equiflow](https://github.com/MoreiraP12/equiflow-v2).

**Pasos que se documentan:**
1. Dataset preoperatorio completo (`OPERA_PRE.xlsx`)
2. Después del merge con datos postoperatorios (`OPERA_COMPLETO.xlsx`)
3. Después del filtro de adultos (`Edad >= 18`)

**Variables trackeadas:**
- `Edad` (continua)
- `Sexo` (categórica: Femenino / Masculino)
- `Tipo de anestesia` (categórica: general, raquídea, sedación, etc.)
- `Mallampati` (categórica: I, II, III / sin dato)
- `IMC` (continua)
- `Necesita valoración` (target binario: prevalencia de la variable objetivo)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('..') 
OUTPUT_DIR = ROOT / 'outputs' / 'cohort_flow'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import os, shutil

# Agrega Graphviz al PATH si no está disponible (común en Windows tras instalación)
_GRAPHVIZ_CANDIDATES = [
    r"C:\Program Files\Graphviz\bin",
    r"C:\Program Files (x86)\Graphviz\bin",
    r"C:\Program Files\Graphviz 2.38\bin",
]
if shutil.which('dot') is None:
    for _p in _GRAPHVIZ_CANDIDATES:
        if os.path.isfile(os.path.join(_p, 'dot.exe')):
            os.environ['PATH'] = _p + os.pathsep + os.environ['PATH']
            print(f'Graphviz encontrado y agregado al PATH: {_p}')
            break
    else:
        raise EnvironmentError(
            'dot.exe no encontrado. Instala Graphviz desde https://graphviz.org/download/ '
            'y reinicia el kernel.'
        )
else:
    print('Graphviz disponible:', shutil.which('dot'))

Graphviz encontrado y agregado al PATH: C:\Program Files\Graphviz\bin


## 1. Carga y preparación de los datos

In [3]:
# --- Paso 0: Dataset crudo preoperatorio ---
df_pre = pd.read_excel(ROOT / 'OPERA_PRE.xlsx')
print(f'OPERA_PRE: {len(df_pre):,} pacientes')

# --- Paso 1: Después del merge (pacientes con datos pre Y postoperatorios) ---
df_completo = pd.read_excel(ROOT / 'OPERA_COMPLETO.xlsx')
print(f'OPERA_COMPLETO (post-merge): {len(df_completo):,} pacientes')
print(f'  Excluidos sin datos postop: {len(df_pre) - len(df_completo):,} ({(len(df_pre) - len(df_completo)) / len(df_pre) * 100:.1f}%)')

# --- Paso 2: Filtro adultos ---
df_adultos = df_completo[df_completo['Edad'] >= 18].reset_index(drop=True)
print(f'Solo adultos (Edad >= 18): {len(df_adultos):,} pacientes')
print(f'  Excluidos pediátricos: {len(df_completo) - len(df_adultos):,} ({(len(df_completo) - len(df_adultos)) / len(df_completo) * 100:.1f}%)')

OPERA_PRE: 30,962 pacientes
OPERA_COMPLETO (post-merge): 29,865 pacientes
  Excluidos sin datos postop: 1,097 (3.5%)
Solo adultos (Edad >= 18): 23,387 pacientes
  Excluidos pediátricos: 6,478 (21.7%)


## 2. Preparación de variables para Equiflow

Equiflow requiere columnas con valores legibles. Reconstruimos variables categóricas desde sus codificaciones.

In [4]:
ANESTESIA_COLS = [
    'Tipo de anestesia propuesta_general',
    'Tipo de anestesia propuesta_raquidea',
    'Tipo de anestesia propuesta_sedacion',
    'Tipo de anestesia propuesta_local',
    'Tipo de anestesia propuesta_peridural',
    'Tipo de anestesia propuesta_bloqueo n',
    'Tipo de anestesia propuesta_bloqueo iv',
    'Tipo de anestesia propuesta_sin dato',
]

LABEL_MAP_ANESTESIA = {
    'Tipo de anestesia propuesta_general': 'General',
    'Tipo de anestesia propuesta_raquidea': 'Raquídea',
    'Tipo de anestesia propuesta_sedacion': 'Sedación',
    'Tipo de anestesia propuesta_local': 'Local',
    'Tipo de anestesia propuesta_peridural': 'Peridural',
    'Tipo de anestesia propuesta_bloqueo n': 'Bloqueo nervioso',
    'Tipo de anestesia propuesta_bloqueo iv': 'Bloqueo IV',
    'Tipo de anestesia propuesta_sin dato': 'Sin dato',
}

LABEL_MAP_MALLAMPATI = {0: 'Sin dato', 1: 'I', 2: 'II', 3: 'III'}
LABEL_MAP_SEXO = {0: 'Femenino', 1: 'Masculino'}
LABEL_MAP_TARGET = {0: 'No requiere', 1: 'Requiere valoración'}


def prepare_for_equiflow(df: pd.DataFrame, has_target: bool = True) -> pd.DataFrame:
    """Añade columnas legibles para Equiflow al DataFrame dado."""
    out = df.copy()

    # Sexo
    out['Sexo'] = out['Sexo_encoded'].map(LABEL_MAP_SEXO)

    # Mallampati
    out['Mallampati'] = out['Puntaje Mallampati'].map(LABEL_MAP_MALLAMPATI)

    # Tipo de anestesia: reconstruir desde one-hot
    available_anest = [c for c in ANESTESIA_COLS if c in out.columns]
    if available_anest:
        out['Tipo de anestesia'] = (
            out[available_anest]
            .idxmax(axis=1)
            .map(LABEL_MAP_ANESTESIA)
        )
        # Filas donde todas las columnas son 0 -> 'Sin dato'
        no_anest = out[available_anest].sum(axis=1) == 0
        out.loc[no_anest, 'Tipo de anestesia'] = 'Sin dato'

    # Target (solo existe en OPERA_COMPLETO y df_adultos)
    if has_target and 'target' in out.columns:
        out['Necesita valoración'] = out['target'].map(LABEL_MAP_TARGET)
    elif has_target:
        # En OPERA_PRE no hay target aún: marcar como NaN
        out['Necesita valoración'] = None

    return out


df_pre_eq = prepare_for_equiflow(df_pre, has_target=False)
df_completo_eq = prepare_for_equiflow(df_completo, has_target=True)
df_adultos_eq = prepare_for_equiflow(df_adultos, has_target=True)

print('Variables preparadas.')
print('Distribución Tipo de anestesia (COMPLETO):')
print(df_completo_eq['Tipo de anestesia'].value_counts())
print('\nPrevalencia target (COMPLETO):', df_completo_eq['Necesita valoración'].value_counts().to_dict())
print('Prevalencia target (Adultos):  ', df_adultos_eq['Necesita valoración'].value_counts().to_dict())

Variables preparadas.
Distribución Tipo de anestesia (COMPLETO):
Tipo de anestesia
General             22881
Sedación             3150
Sin dato             2743
Raquídea             1074
Local                   8
Bloqueo nervioso        6
Peridural               2
Bloqueo IV              1
Name: count, dtype: int64

Prevalencia target (COMPLETO): {'Requiere valoración': 17475, 'No requiere': 12390}
Prevalencia target (Adultos):   {'Requiere valoración': 14022, 'No requiere': 9365}


## 3. Generación del diagrama Equiflow

In [5]:
from equiflow import EquiFlow

# Columnas de referencia comunes a los tres stages
TRACK_COLS_EQ = ['Sexo', 'Tipo de anestesia', 'Mallampati']
TRACK_CONTINUOUS = ['Edad', 'IMC']

# Inicializar con dataset inicial (OPERA_PRE)
ef = EquiFlow(
    data=df_pre_eq,
    initial_cohort_label="Pacientes quirúrgicos (OPERA_PRE)",
    categorical=TRACK_COLS_EQ,
    nonnormal=TRACK_CONTINUOUS,        # parámetro correcto: nonnormal
    limit=5,                           # máximo 5 categorías por variable (evita gráficas saturadas)
)

# Paso 1: Exclusión por falta de datos postoperatorios (merge)
ef.add_exclusion(
    new_cohort=df_completo_eq,         # pasar el DataFrame filtrado con new_cohort=
    exclusion_reason="sin datos postoperatorios",
    new_cohort_label="Con seguimiento postoperatorio",
)

# Paso 2: Exclusión de pacientes pediátricos
ef.add_exclusion(
    new_cohort=df_adultos_eq,
    exclusion_reason="pacientes pediátricos (Edad < 18)",
    new_cohort_label="Solo adultos (Edad ≥ 18)",
)

print('Equiflow inicializado correctamente.')

Equiflow inicializado correctamente.


In [6]:
# Tabla de características por paso
df_chars = ef.view_table_characteristics()
display(df_chars)

Cohort                     \
                                                    0                  1   
Variable                 Value                                             
Overall                                        30,962             29,865   
Sexo, N (%)              Masculino      17,930 (57.9)      17,370 (58.2)   
                         Femenino       13,032 (42.1)      12,495 (41.8)   
                         Missing              0 (0.0)            0 (0.0)   
Tipo de anestesia, N (%) General        23,474 (75.8)      22,881 (76.6)   
                         Sedación        3,380 (10.9)       3,150 (10.5)   
                         Sin dato         2,983 (9.6)        2,743 (9.2)   
                         Raquídea         1,106 (3.6)        1,074 (3.6)   
                         Local                9 (0.0)            8 (0.0)   
                         Missing              0 (0.0)            0 (0.0)   
Mallampati, N (%)        Sin dato       19,955 (64.4)      19,173 (64.2)   
                         I               9,352 (30.2)       9,065 (30.4)   
                         II               1,330 (4.3)        1,303 (4.4)   
                         III                325 (1.0)          324 (1.1)   
                         Missing              0 (0.0)            0 (0.0)   
Edad, Median [IQR]                  38.0 [22.0, 54.0]  38.0 [22.0, 54.0]   
                         Missing              0 (0.0)            0 (0.0)   
IMC, Median [IQR]                   24.7 [21.3, 27.5]  24.7 [21.2, 27.5]   
                         Missing              0 (0.0)            0 (0.0)   

                                                       
                                                    2  
Variable                 Value                         
Overall                                        23,387  
Sexo, N (%)              Masculino      14,434 (61.7)  
                         Femenino        8,953 (38.3)  
                         Missing              0 (0.0)  
Tipo de anestesia, N (%) General        17,474 (74.7)  
                         Sedación        2,529 (10.8)  
                         Sin dato         2,301 (9.8)  
                         Raquídea         1,068 (4.6)  
                         Local                8 (0.0)  
                         Missing              0 (0.0)  
Mallampati, N (%)        Sin dato       13,858 (59.3)  
                         I               7,977 (34.1)  
                         II               1,247 (5.3)  
                         III                305 (1.3)  
                         Missing              0 (0.0)  
Edad, Median [IQR]                  44.0 [34.0, 57.0]  
                         Missing              0 (0.0)  
IMC, Median [IQR]                   26.0 [23.5, 28.0]  
                         Missing              0 (0.0)

In [7]:
# Tabla de drift (SMDs entre pasos)
df_drift = ef.view_table_drifts()
display(df_drift)

Cohort Flow,0 to 1,1 to 2
Sexo,0.01,0.07
Tipo de anestesia,0.02,0.06
Mallampati,0.02,0.1
Edad,-0.0,0.44
IMC,-0.0,0.4


In [8]:
# Generar y guardar el diagrama de flujo (3 pasos, sin target)
ef.plot_flows(
    smds=True,
    legend=True,
    smd_decimals=2,
    output_folder=str(OUTPUT_DIR),
    output_file="cohort_flow_completo",
)
print(f'Diagrama guardado en: {OUTPUT_DIR / "cohort_flow_completo.pdf"}')

Diagrama guardado en: ..\outputs\cohort_flow\cohort_flow_completo.pdf


## 4. Versión con target (solo en pacientes con datos postop)

Esta sección muestra cómo cambia la **prevalencia de necesidad de valoración preanestésica** al aplicar el filtro de adultos.

In [9]:
# Segunda instancia de Equiflow: desde el dataset mergeado hacia adultos,
# ahora incluyendo la variable target para ver cómo cambia la prevalencia.

ef2 = EquiFlow(
    data=df_completo_eq,
    initial_cohort_label="Con seguimiento postoperatorio",
    categorical=TRACK_COLS_EQ + ['Necesita valoración'],
    nonnormal=TRACK_CONTINUOUS,
    limit=5,
)

ef2.add_exclusion(
    new_cohort=df_adultos_eq,
    exclusion_reason="pacientes pediátricos (Edad < 18)",
    new_cohort_label="Solo adultos (Edad ≥ 18)",
)

ef2.plot_flows(
    smds=True,
    legend=True,
    smd_decimals=2,
    output_folder=str(OUTPUT_DIR),
    output_file="cohort_flow_con_target",
)
print(f'Diagrama guardado en: {OUTPUT_DIR / "cohort_flow_con_target.pdf"}')

Diagrama guardado en: ..\outputs\cohort_flow\cohort_flow_con_target.pdf


## 5. Exportar tablas auxiliares

In [ ]:
# Tabla resumen del flujo (n por paso)
df_flow = ef.view_table_flows()
display(df_flow)

# Guardar como CSV
df_chars.to_csv(OUTPUT_DIR / 'characteristics_table.csv')
df_drift.to_csv(OUTPUT_DIR / 'drift_smd_table.csv')
df_flow.to_csv(OUTPUT_DIR / 'flow_summary.csv')

# LaTeX
try:
    df_chars.to_latex(OUTPUT_DIR / 'characteristics_table.tex', escape=False)
    df_drift.to_latex(OUTPUT_DIR / 'drift_smd_table.tex', escape=False)
    print('Tablas LaTeX exportadas.')
except Exception as e:
    print(f'LaTeX export skipped: {e}')

print(f'Archivos guardados en: {OUTPUT_DIR.resolve()}')

Cohort Flow,0 to 1,1 to 2
,,
"Initial, n","30,962","29,865"
"Removed, n","1,097","6,478"
"Result, n","29,865","23,387"


Tablas LaTeX exportadas.
Archivos guardados en: C:\Users\Usuario\Desktop\predictive-screening-preanesthesia\outputs\cohort_flow
